# News NLP pipeline — small runnable demo

This notebook follows a handful of synthetic headlines through the same stages used by the prototype. There is no live news retrieval, and the sample does not state real events about real companies.

![News NLP pipeline](../docs/news_nlp/pipeline_overview.svg)

By default, tiny deterministic test doubles exercise the plumbing without downloading model weights. Set `RUN_REAL_INFERENCE = True` to use FinBERT-ESG, the MiniLM financial fallback and our fine-tuned direction checkpoint.

In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise FileNotFoundError("Start Jupyter in the repository root or notebooks/ folder.")

sys.path.insert(0, str(REPO_ROOT))
from src.news_nlp import (
    NewsPipeline, aggregate_event_features, build_default_pipeline,
    compute_tone_momentum,
)

## 1. Load the demonstration headlines

These ten rows were written for this repository. Two rows deliberately repeat one headline so we can see that duplicate coverage does not create two direction votes.

In [2]:
AS_OF_UTC = "2026-09-13T00:00:00Z"
RUN_REAL_INFERENCE = False
DIRECTION_MODEL_PATH_OR_ID = os.environ.get("DIRECTION_MODEL_PATH_OR_ID", "")

news = pd.read_csv(REPO_ROOT / "data" / "news_demo_headlines.csv")
assert news["is_synthetic"].eq(1).all()
assert news[["company_id", "company_name", "headline", "published_at_utc"]].notna().all().all()

display(news[["company_name", "headline", "published_at_utc"]])

,company_name,headline,published_at_utc
0,Northstar Energy,Demo: Northstar Energy begins operating a new ...,2026-09-08T09:00:00Z
1,Northstar Energy,Demo: Northstar Energy begins operating a new ...,2026-09-08T13:00:00Z
2,Northstar Energy,Demo: Northstar Energy reports a temporary inc...,2026-07-22T11:00:00Z
3,Harbor Bank,Demo: Harbor Bank reports quarterly profit in ...,2026-09-04T08:30:00Z
4,Harbor Bank,Demo: Harbor Bank raises its loan-loss provisi...,2026-06-18T15:00:00Z
5,Orbit Foods,Demo: Orbit Foods recalls a product after a cu...,2026-09-10T07:00:00Z
6,Orbit Foods,Demo: Orbit Foods expands paid parental leave ...,2026-08-11T10:00:00Z
7,Cedar Software,Demo: Cedar Software appoints two independent ...,2026-09-01T12:00:00Z
8,Cedar Software,Demo: A regulator opens an inquiry into Cedar ...,2026-05-30T14:00:00Z
9,Cedar Software,Demo: Cedar Software publishes its annual meet...,2026-09-12T16:00:00Z


## 2. Choose real models or a plumbing check

The test doubles below are deliberately simple keyword rules. Their outputs are **not model results and not an accuracy claim**; they only let reviewers run every pipeline step offline. The real path uses the three model layers.

In [3]:
class DemoESGClassifier:
    # Deterministic, plumbing-only replacement for FinBERT-ESG.
    def predict_proba(self, texts):
        answers = []
        for text in texts:
            text = text.lower()
            if any(word in text for word in ("solar", "emissions")):
                top = "environmental"
            elif any(word in text for word in ("safety", "parental leave")):
                top = "social"
            elif any(word in text for word in ("directors", "disclosure", "meeting")):
                top = "governance"
            else:
                top = "none"
            result = {label: 0.025 for label in ("environmental", "social", "governance", "none")}
            result[top] = 0.925
            answers.append(result)
        return answers


class DemoFinancialFallback:
    # Plumbing-only replacement for the MiniLM fallback.
    def score(self, texts):
        terms = ("profit", "loan-loss", "defaults")
        return [0.85 if any(term in text.lower() for term in terms) else None for text in texts]


class DemoDirectionClassifier:
    # Plumbing-only replacement for the fine-tuned direction model.
    def predict_proba(self, texts):
        answers = []
        for text in texts:
            text = text.lower()
            negative = ("increase in", "raises", "recalls", "inquiry")
            positive = ("begins operating", "expands", "appoints")
            top = "negative" if any(word in text for word in negative) else (
                "positive" if any(word in text for word in positive) else "neutral"
            )
            result = {label: 0.10 for label in ("negative", "neutral", "positive")}
            result[top] = 0.80
            answers.append(result)
        return answers

In [4]:
if RUN_REAL_INFERENCE:
    if not DIRECTION_MODEL_PATH_OR_ID:
        raise ValueError("Set DIRECTION_MODEL_PATH_OR_ID to the selected checkpoint.")
    pipeline = build_default_pipeline(DIRECTION_MODEL_PATH_OR_ID, device="auto")
    run_label = "real model inference"
else:
    pipeline = NewsPipeline(
        esg_classifier=DemoESGClassifier(),
        financial_fallback=DemoFinancialFallback(),
        direction_classifier=DemoDirectionClassifier(),
    )
    run_label = "offline plumbing check — test doubles, not model predictions"

print(run_label)

offline plumbing check — test doubles, not model predictions


## 3. Classify and inspect the trace

Company names are masked before topic classification. FinBERT-ESG chooses Environmental, Social, Governance or None. Only None can enter the conservative financial fallback. The assigned pillar then becomes part of the direction model's input.

In [5]:
scored = pipeline.classify(news, deduplicate=True)

trace_columns = [
    "company_name", "headline", "article_count", "esg_top_label",
    "pillar_label", "pillar_method", "financial_fallback_score",
    "direction_label", "p_negative", "p_neutral", "p_positive",
    "signed_tone", "inference_status",
]
display(scored[trace_columns].round(3))

assert len(scored) == len(news) - 1  # the exact duplicate became one event
assert scored["signed_tone"].dropna().between(-1, 1).all()
fallback_rows = scored["pillar_method"].eq("minilm_financial_fallback")
assert fallback_rows.any()
assert scored.loc[fallback_rows, "esg_top_label"].eq("none").all()

,company_name,headline,article_count,esg_top_label,pillar_label,pillar_method,financial_fallback_score,direction_label,p_negative,p_neutral,p_positive,signed_tone,inference_status
0,Cedar Software,Demo: A regulator opens an inquiry into Cedar ...,1,governance,governance,finbert_esg,NaN,negative,0.8,0.1,0.1,-0.7,direction_scored
1,Cedar Software,Demo: Cedar Software appoints two independent ...,1,governance,governance,finbert_esg,NaN,positive,0.1,0.1,0.8,0.7,direction_scored
2,Cedar Software,Demo: Cedar Software publishes its annual meet...,1,governance,governance,finbert_esg,NaN,neutral,0.1,0.8,0.1,0.0,direction_scored
3,Harbor Bank,Demo: Harbor Bank raises its loan-loss provisi...,1,none,financial,minilm_financial_fallback,0.85,negative,0.8,0.1,0.1,-0.7,direction_scored
4,Harbor Bank,Demo: Harbor Bank reports quarterly profit in ...,1,none,financial,minilm_financial_fallback,0.85,neutral,0.1,0.8,0.1,0.0,direction_scored
5,Northstar Energy,Demo: Northstar Energy reports a temporary inc...,1,environmental,environmental,finbert_esg,NaN,negative,0.8,0.1,0.1,-0.7,direction_scored
6,Northstar Energy,Demo: Northstar Energy begins operating a new ...,2,environmental,environmental,finbert_esg,NaN,positive,0.1,0.1,0.8,0.7,direction_scored
7,Orbit Foods,Demo: Orbit Foods expands paid parental leave ...,1,social,social,finbert_esg,NaN,positive,0.1,0.1,0.8,0.7,direction_scored
8,Orbit Foods,Demo: Orbit Foods recalls a product after a cu...,1,social,social,finbert_esg,NaN,negative,0.8,0.1,0.1,-0.7,direction_scored


## 4. Build company features

For each company, pillar and window, the feature builder counts deduplicated events and computes weighted tone. Direction is

$$s_i=P_i(\text{positive})-P_i(\text{negative}),$$

and recent evidence receives more weight. Thin evidence is shrunk toward zero, while raw tone remains missing when there is no evidence.

In [6]:
features = aggregate_event_features(
    scored,
    company_ids=news["company_id"].unique(),
    as_of_utc=AS_OF_UTC,
    windows=(30, 90, 180),
)
momentum = compute_tone_momentum(features)

summary_90d = features.loc[
    features["window_days"].eq(90) & features["event_count"].gt(0),
    ["company_id", "pillar", "event_count", "article_count", "effective_weight",
     "tone_raw", "tone_shrunk_to_zero"],
].sort_values(["company_id", "pillar"])

display(summary_90d.round(3))
display(momentum.loc[momentum["baseline_window_has_evidence"].eq(1)].round(3))
assert len(features) == news["company_id"].nunique() * 4 * 3

,company_id,pillar,event_count,article_count,effective_weight,tone_raw,tone_shrunk_to_zero
31,demo:cedar,governance,2,2,1.609,0.320,0.172
20,demo:harbor,financial,2,2,0.902,-0.096,-0.029
17,demo:northstar,environmental,2,3,1.136,0.363,0.138
26,demo:orbit,social,2,2,1.349,-0.245,-0.110


,company_id,pillar,short_window_days,baseline_window_days,tone_momentum,short_window_has_evidence,baseline_window_has_evidence
2,demo:cedar,governance,30,90,0.000,1,1
5,demo:harbor,financial,30,90,0.029,1,1
8,demo:northstar,environmental,30,90,0.064,1,1
15,demo:orbit,social,30,90,-0.102,1,1


## Reading the output

- `pillar_method` shows whether FinBERT-ESG or the financial fallback assigned the topic.
- `signed_tone` is continuous: negative values point toward adverse news and positive values toward favorable news.
- Counts describe the collected sample; they are not company ESG performance.
- This notebook proves the inference and aggregation path. The full prototype still needs dated retrieval, relevance review and coverage diagnostics before its features are merged with structured ESG data.